In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

In [2]:
def calculate_idx_position(account_capital_idr, risk_pct, entry_price, stop_loss_price):

    risk_amount_idr = account_capital_idr * (risk_pct / 100)

    if stop_loss_price <= 1:
      new_stop_loss_price = entry_price * (1-stop_loss_price)
      risk_per_share = entry_price - new_stop_loss_price
    else:
      risk_per_share = entry_price - stop_loss_price

    total_shares = risk_amount_idr / risk_per_share

    if total_shares < 100:
      total_lots = 0
    else :
      total_lots = int(total_shares // 100)

    position_value_idr = total_lots * 100 * entry_price

    print(f"--- IDX RISK MANAGEMENT REPORT ---")
    print(f"Account Capital : Rp {account_capital_idr:,.0f}")
    print(f"Stop Loss price : Rp {new_stop_loss_price}")
    print(f"Max Risk Allowed: Rp {risk_amount_idr:,.0f} ({risk_pct}%)")
    if total_lots == 0:
      print("Suggested Buy   : Lots are Low")
    else:
      print(f"Suggested Buy   : {total_lots} Lot(s) ({total_lots * 100} shares)")
    print(f"Capital Deployed: Rp {position_value_idr:,.0f}")

    return total_lots

In [16]:
def calculate_phase2_ema_strategy(df, ema_short=15, ema_long=30, rsi_period=15):

    df['EMA_Short'] = df['Close'].ewm(span=ema_short, adjust=False).mean()
    df['EMA_Long'] = df['Close'].ewm(span=ema_long, adjust=False).mean()

    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.ewm(alpha=1/rsi_period, min_periods=rsi_period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/rsi_period, min_periods=rsi_period, adjust=False).mean()

    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))
    df['Signal'] = 0
    momentum_bullish = df['EMA_Short'] > df['EMA_Long']
    healthy_momentum = (df['RSI'] > 50) & (df['RSI'] < 70)
    df.loc[momentum_bullish & healthy_momentum, 'Signal'] = 1

    return df

In [17]:
#Phase 3: master Test case
if __name__ == "__main__":
    ticker = "ADRO.JK"
    capital = 10000000  # Rp 10,000,000 account capital
    risk_tolerance = 1.0 # 1% risk per trade
    stop_loss_pct = 0.05 # 5% stop loss drop

    print(f"Fetching market data and running pipeline for {ticker}...")
    df = yf.download(ticker, period="1y", interval="1d")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)

    # Step 1: Run indicators and signals
    processed_df = calculate_phase2_ema_strategy(df)

    # Step 2: Check the latest market day
    latest_row = processed_df.iloc[-1]
    latest_close = latest_row['Close']
    latest_signal = latest_row['Signal']
    latest_rsi = latest_row['RSI']

    print(f"\n--- MARKET STATUS REPORT ---")
    print(f"Latest Date   : {processed_df.index[-1].strftime('%Y-%m-%d')}")
    print(f"Latest Close  : Rp {latest_close:,.2f}")
    print(f"Latest RSI    : {latest_rsi:.2f}")
    print(f"Model Signal  : {int(latest_signal)}")

    # Step 3: Conditional automated execution
    if latest_signal == 1:
        print("Status: Bullish Confluence Met! Calculating position size...")
        calculate_idx_position(
            account_capital_idr=capital,
            risk_pct=risk_tolerance,
            entry_price=latest_close,
            stop_loss_price=stop_loss_pct
        )
    else:
        print("Status: No trade triggered (Market is bearish, flat, or RSI is overbought). Stand down.")

/tmp/ipykernel_523/581522479.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="1y", interval="1d")
[*********************100%***********************]  1 of 1 completed

Fetching market data and running pipeline for ADRO.JK...

--- MARKET STATUS REPORT ---
Latest Date   : 2026-10-08
Latest Close  : Rp 2,550.00
Latest RSI    : 47.67
Model Signal  : 0
Status: No trade triggered (Market is bearish, flat, or RSI is overbought). Stand down.


In [18]:
# --- HISTORICAL SIGNAL SCANNER ---
# Filter the dataframe to show only rows where Signal was 1
historical_buys = processed_df[processed_df['Signal'] == 1]

print(f"\n--- TOTAL BUY SIGNALS FOUND PAST YEAR: {len(historical_buys)} ---")
if len(historical_buys) > 0:
    print(historical_buys[['Close', 'EMA_Short', 'EMA_Long', 'RSI']].tail(10))
else:
    print("No buy signals found with these parameters over the past year. Try a different stock or looser indicator spans!")


--- TOTAL BUY SIGNALS FOUND PAST YEAR: 132 ---
Price        Close    EMA_Short     EMA_Long        RSI
Date                                                   
2026-09-03  2740.0  2648.898216  2581.498596  60.300155
2026-09-04  2720.0  2657.785939  2590.434170  58.737101
2026-09-07  2700.0  2663.062697  2597.502934  57.149892
2026-09-08  2690.0  2666.429860  2603.470486  56.334384
2026-09-09  2690.0  2669.376127  2609.053036  56.334384
2026-09-10  2670.0  2669.454111  2612.985098  54.547308
2026-09-11  2640.0  2665.772347  2614.727995  51.901238
2026-09-14  2620.0  2660.050804  2615.068124  50.163099
2026-09-17  2670.0  2648.022316  2616.287351  54.634716
2026-09-18  2640.0  2647.019526  2617.817199  51.816630
